# AeroSentinel — Complete Training Pipeline
## Intelligent Engine Health Monitoring & Predictive Maintenance
**Dataset:** NASA C-MAPSS Turbofan Engine Degradation — FD001

### What this notebook trains:
| Model | Type | Purpose |
|---|---|---|
| Random Forest | Regression | Predict RUL (remaining useful life) |
| Gradient Boosting | Regression | Predict RUL |
| XGBoost | Regression | Predict RUL |
| Random Forest | Classification | Failure within 30 cycles? |
| Gradient Boosting | Classification | Failure within 30 cycles? |
| LSTM | Deep Learning | Predict RUL from 30-cycle sequences |
| GRU | Deep Learning | Predict RUL from 30-cycle sequences |
| Isolation Forest | Anomaly Detection | Detect abnormal sensor patterns |

### Outputs:
- All trained models saved as `.joblib` / `.keras`
- **Table 3** — complete measured metrics (MAE, RMSE, R²)
- **Figs 1–8** — all publication-quality figures
- Everything downloadable as a ZIP

---
**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All (Ctrl+F9)
3. Download `AeroSentinel_Results.zip` from the last cell

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install xgboost shap scikit-learn pandas numpy matplotlib seaborn joblib tensorflow

## Cell 2 — Upload dataset files
Upload these 7 files from your `NASATurbofanEnginePredictiveMaintenance/` folder:
- `train_FD001.txt`
- `test_FD001.txt`
- `RUL_FD001.txt`

Or mount Google Drive if you have them there.

In [ ]:
import os

# ── OPTION A: Upload files manually ──────────────────────────────────────────
from google.colab import files
print('Upload train_FD001.txt, test_FD001.txt, RUL_FD001.txt')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

# ── OPTION B: Mount Google Drive (comment out Option A above) ────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_PATH = '/content/drive/MyDrive/CMaps/'   # ← change to your folder
# import shutil
# for f in ['train_FD001.txt','test_FD001.txt','RUL_FD001.txt']:
#     shutil.copy(DRIVE_PATH + f, f)
# print('Copied from Drive')

## Cell 3 — Imports & global config

In [ ]:
import json, os, warnings, time
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               RandomForestClassifier, GradientBoostingClassifier,
                               IsolationForest)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              precision_score, recall_score, f1_score, roc_auc_score,
                              confusion_matrix, roc_curve)
import xgboost as xgb
import shap
import joblib

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('NumPy:', np.__version__)

# ── Output directory ──────────────────────────────────────────────────────────
OUT    = Path('results_figures'); OUT.mkdir(exist_ok=True)
MODELS = Path('models');          MODELS.mkdir(exist_ok=True)

# ── Colour palette ────────────────────────────────────────────────────────────
BG  = '#070B16'; BG2 = '#0B1220'; SURF = '#111B2F'
PRI = '#69C9FF'; SEC = '#8B83FF'; ACC  = '#36E0C1'
WRN = '#FFBF69'; DNG = '#FF647C'; SUC  = '#39D98A'
TXT = '#F4F7FF'; TXT2= '#9AAAC2'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG2,
    'axes.edgecolor': '#1E2D4A', 'axes.labelcolor': TXT2,
    'axes.titlecolor': TXT, 'xtick.color': TXT2,
    'ytick.color': TXT2, 'text.color': TXT,
    'grid.color': '#1E2D4A', 'grid.alpha': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('✓ Setup complete')

## Cell 4 — Data loading & feature engineering

In [ ]:
# ── Column schema ─────────────────────────────────────────────────────────────
INDEX    = ['unit','cycle']
SETTINGS = ['setting1','setting2','setting3']
SENSORS  = [f'sensor{i}' for i in range(1,22)]
COLS     = INDEX + SETTINGS + SENSORS

def load_raw(path):
    return (pd.read_csv(path, sep=r'\s+', header=None, names=COLS)
              .sort_values(INDEX).reset_index(drop=True))

def add_rul_target(df, cap=None):
    mc = df.groupby('unit')['cycle'].max().reset_index()
    mc.columns = ['unit','max_cycle']
    df = df.merge(mc, on='unit', how='left')
    df['RUL'] = df['max_cycle'] - df['cycle']
    if cap: df['RUL'] = df['RUL'].clip(upper=cap)
    return df.drop('max_cycle', axis=1)

def add_rolling(df, ws=[5,10]):
    cols = {}
    for w in ws:
        for s in SENSORS:
            cols[f'{s}_rolling_mean_{w}'] = df.groupby('unit')[s].transform(
                lambda x: x.rolling(w, min_periods=1).mean())
            cols[f'{s}_rolling_std_{w}']  = df.groupby('unit')[s].transform(
                lambda x: x.rolling(w, min_periods=1).std()).fillna(0)
    return pd.concat([df, pd.DataFrame(cols, index=df.index)], axis=1)

def add_deg(df):
    mc = df.groupby('unit')['cycle'].max().reset_index()
    mc.columns = ['unit','max_cycle']
    df = df.merge(mc, on='unit', how='left')
    cols = {}
    for s in SENSORS:
        cols[f'{s}_diff_from_start'] = df.groupby('unit')[s].transform(
            lambda x: x - x.iloc[0])
        cols[f'{s}_normalized'] = df[s] / (df['max_cycle'] + 1)
    df = pd.concat([df, pd.DataFrame(cols, index=df.index)], axis=1)
    return df.drop('max_cycle', axis=1)

def engineer(df, rul_cap=None):
    df = df.copy()
    df = add_rul_target(df, cap=rul_cap)
    df = add_rolling(df)
    df = add_deg(df)
    df['failure_within_30'] = (df['RUL'] <= 30).astype(int)
    return df

EXCL = {'unit','cycle','RUL','failure_within_30'}

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading FD001 ...')
train_raw = load_raw('train_FD001.txt')
test_raw  = load_raw('test_FD001.txt')
rul_true  = pd.read_csv('RUL_FD001.txt', sep=r'\s+',
                         header=None, names=['RUL'])['RUL']

print(f'  train: {train_raw.shape} | test: {test_raw.shape} | rul entries: {len(rul_true)}')

# ── Engineer features (sklearn models — uncapped RUL) ─────────────────────────
train_eng = engineer(train_raw)
test_eng  = engineer(test_raw)

FEAT_COLS = [c for c in train_eng.columns if c not in EXCL]
print(f'  Feature count: {len(FEAT_COLS)}')

# ── Test: last cycle per engine ───────────────────────────────────────────────
last_idx   = test_eng.groupby('unit')['cycle'].idxmax()
test_last  = test_eng.loc[last_idx].sort_values('unit').reset_index(drop=True)
test_last['RUL_true'] = rul_true.values

X_train = train_eng[FEAT_COLS].values
y_rul   = train_eng['RUL'].values
y_fail  = train_eng['failure_within_30'].values
X_test  = test_last[FEAT_COLS].values
y_test  = rul_true.values
y_test_fail = (rul_true.values <= 30).astype(int)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

print(f'  X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'  Positive rate (failure_within_30): {y_fail.mean():.3f}')

# Verify no leakage
assert 'RUL' not in FEAT_COLS
assert 'failure_within_30' not in FEAT_COLS
print('✓ No target leakage confirmed')

## Cell 5 — Train sklearn models (RF, GB, XGBoost)

In [ ]:
def reg_metrics(name, model, X, y):
    p = np.maximum(model.predict(X), 0)
    mae  = float(mean_absolute_error(y, p))
    rmse = float(np.sqrt(mean_squared_error(y, p)))
    r2   = float(r2_score(y, p))
    print(f'  {name:<28} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'preds': p}

def cls_metrics(name, model, X, y):
    p    = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    pr   = float(precision_score(y, p))
    rc   = float(recall_score(y, p))
    f1   = float(f1_score(y, p))
    auc  = float(roc_auc_score(y, prob))
    print(f'  {name:<28} P={pr:.4f}  R={rc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    return {'precision': pr, 'recall': rc, 'f1': f1, 'roc_auc': auc,
            'preds': p, 'proba': prob}

# ── RUL Regression ────────────────────────────────────────────────────────────
print('Training RUL regression models ...')
t0 = time.time()

rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_tr_sc, y_rul)
rf_r = reg_metrics('Random Forest', rf, X_te_sc, y_test)

gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
gb.fit(X_tr_sc, y_rul)
gb_r = reg_metrics('Gradient Boosting', gb, X_te_sc, y_test)

xgb_m = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           random_state=42, n_jobs=-1, verbosity=0)
xgb_m.fit(X_tr_sc, y_rul)
xgb_r = reg_metrics('XGBoost', xgb_m, X_te_sc, y_test)

print(f'  Elapsed: {time.time()-t0:.1f}s')

# ── Failure Classification ────────────────────────────────────────────────────
print('\nTraining failure classifiers ...')
t0 = time.time()

fail_rf = RandomForestClassifier(n_estimators=100, max_depth=15,
                                  random_state=42, n_jobs=-1)
fail_rf.fit(X_tr_sc, y_fail)
failrf_r = cls_metrics('RF Classifier', fail_rf, X_te_sc, y_test_fail)

fail_gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
fail_gb.fit(X_tr_sc, y_fail)
failgb_r = cls_metrics('GB Classifier (preferred)', fail_gb, X_te_sc, y_test_fail)

print(f'  Elapsed: {time.time()-t0:.1f}s')

# ── Save sklearn artifacts ────────────────────────────────────────────────────
joblib.dump(rf,       MODELS / 'rul_rf_fd001.joblib')
joblib.dump(gb,       MODELS / 'rul_gb_fd001.joblib')
joblib.dump(xgb_m,    MODELS / 'rul_xgb_fd001.joblib')
joblib.dump(fail_rf,  MODELS / 'failure_rf_fd001.joblib')
joblib.dump(fail_gb,  MODELS / 'failure_gb_fd001.joblib')
joblib.dump(scaler,   MODELS / 'scaler_fd001.joblib')
with open(MODELS / 'feature_cols_fd001.json', 'w') as f:
    json.dump(FEAT_COLS, f, indent=2)
print('\n✓ Sklearn models saved')

## Cell 6 — Train LSTM

In [ ]:
# ── LSTM config ───────────────────────────────────────────────────────────────
WINDOW   = 30
RUL_CAP  = 125
EPOCHS   = 50
BATCH    = 128
LR       = 1e-3

# 14 informative sensors (constant/near-zero-variance sensors excluded)
SEQ_SENSORS = ['sensor2','sensor3','sensor4','sensor7','sensor8',
                'sensor9','sensor11','sensor12','sensor13','sensor14',
                'sensor15','sensor17','sensor20','sensor21']
N_FEAT = len(SEQ_SENSORS)

# ── Capped RUL + normalise ────────────────────────────────────────────────────
train_cap = add_rul_target(train_raw.copy(), cap=RUL_CAP)

seq_scaler = MinMaxScaler()
train_cap[SEQ_SENSORS] = seq_scaler.fit_transform(train_cap[SEQ_SENSORS])
test_seq   = test_raw.copy()
test_seq[SEQ_SENSORS]  = seq_scaler.transform(test_seq[SEQ_SENSORS])

# ── Build sequences ───────────────────────────────────────────────────────────
def make_sequences(df, window, features, rul_col='RUL'):
    X_list, y_list = [], []
    for _, grp in df.groupby('unit'):
        data = grp[features].values
        rul  = grp[rul_col].values
        n    = len(grp)
        if n < window:
            pad  = np.repeat(data[[0]], window - n, axis=0)
            data = np.vstack([pad, data])
            rul  = np.concatenate([np.full(window - n, rul[0]), rul])
            n    = window
        for i in range(n - window + 1):
            X_list.append(data[i:i+window])
            y_list.append(rul[i+window-1])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def make_test_sequences(df, window, features):
    X_list = []
    for _, grp in df.groupby('unit'):
        data = grp[features].values
        n    = len(grp)
        if n < window:
            pad  = np.repeat(data[[0]], window - n, axis=0)
            data = np.vstack([pad, data])
        X_list.append(data[-window:])
    return np.array(X_list, dtype=np.float32)

print('Building sequences ...')
X_seq_tr, y_seq_tr = make_sequences(train_cap, WINDOW, SEQ_SENSORS)
X_seq_te           = make_test_sequences(test_seq, WINDOW, SEQ_SENSORS)
y_seq_te           = rul_true.values.astype(np.float32)
print(f'  X_train: {X_seq_tr.shape} | X_test: {X_seq_te.shape}')

# ── Build LSTM model ──────────────────────────────────────────────────────────
lstm_model = Sequential([
    Input(shape=(WINDOW, N_FEAT)),
    LSTM(128, return_sequences=True, activation='tanh'),
    Dropout(0.2),
    LSTM(64, return_sequences=False, activation='tanh'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
], name='lstm_rul')

lstm_model.compile(optimizer=Adam(LR), loss='mse', metrics=['mae'])
lstm_model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-5, verbose=1),
]

print(f'\nTraining LSTM ({EPOCHS} epochs max) ...')
t0 = time.time()
lstm_hist = lstm_model.fit(
    X_seq_tr, y_seq_tr,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1,
)
print(f'  Elapsed: {time.time()-t0:.1f}s | Epochs run: {len(lstm_hist.history["loss"])}')

lstm_preds = np.maximum(lstm_model.predict(X_seq_te, verbose=0).flatten(), 0)
lstm_r = {
    'mae':  float(mean_absolute_error(y_seq_te, lstm_preds)),
    'rmse': float(np.sqrt(mean_squared_error(y_seq_te, lstm_preds))),
    'r2':   float(r2_score(y_seq_te, lstm_preds)),
    'preds': lstm_preds,
    'epochs_run': len(lstm_hist.history['loss'])
}
print(f'  LSTM  MAE={lstm_r["mae"]:.4f}  RMSE={lstm_r["rmse"]:.4f}  R²={lstm_r["r2"]:.4f}')
lstm_model.save(MODELS / 'lstm_fd001.keras')
joblib.dump(seq_scaler, MODELS / 'seq_scaler_fd001.joblib')
print('✓ LSTM saved')

## Cell 7 — Train GRU

In [ ]:
gru_model = Sequential([
    Input(shape=(WINDOW, N_FEAT)),
    GRU(128, return_sequences=True, activation='tanh'),
    Dropout(0.2),
    GRU(64, return_sequences=False, activation='tanh'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
], name='gru_rul')

gru_model.compile(optimizer=Adam(LR), loss='mse', metrics=['mae'])
gru_model.summary()

print(f'\nTraining GRU ({EPOCHS} epochs max) ...')
t0 = time.time()
gru_hist = gru_model.fit(
    X_seq_tr, y_seq_tr,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1,
)
print(f'  Elapsed: {time.time()-t0:.1f}s | Epochs run: {len(gru_hist.history["loss"])}')

gru_preds = np.maximum(gru_model.predict(X_seq_te, verbose=0).flatten(), 0)
gru_r = {
    'mae':  float(mean_absolute_error(y_seq_te, gru_preds)),
    'rmse': float(np.sqrt(mean_squared_error(y_seq_te, gru_preds))),
    'r2':   float(r2_score(y_seq_te, gru_preds)),
    'preds': gru_preds,
    'epochs_run': len(gru_hist.history['loss'])
}
print(f'  GRU   MAE={gru_r["mae"]:.4f}  RMSE={gru_r["rmse"]:.4f}  R²={gru_r["r2"]:.4f}')
gru_model.save(MODELS / 'gru_fd001.keras')
print('✓ GRU saved')

## Cell 8 — Table 3 (complete measured metrics)

In [ ]:
table3 = {
    'Random Forest':     {'MAE': round(rf_r['mae'],4),   'RMSE': round(rf_r['rmse'],4),   'R2': round(rf_r['r2'],4)},
    'Gradient Boosting': {'MAE': round(gb_r['mae'],4),   'RMSE': round(gb_r['rmse'],4),   'R2': round(gb_r['r2'],4)},
    'XGBoost':           {'MAE': round(xgb_r['mae'],4),  'RMSE': round(xgb_r['rmse'],4),  'R2': round(xgb_r['r2'],4)},
    'LSTM':              {'MAE': round(lstm_r['mae'],4),  'RMSE': round(lstm_r['rmse'],4), 'R2': round(lstm_r['r2'],4)},
    'GRU':               {'MAE': round(gru_r['mae'],4),   'RMSE': round(gru_r['rmse'],4),  'R2': round(gru_r['r2'],4)},
}

print('='*60)
print('TABLE 3 — COMPLETE MEASURED METRICS (FD001 Test Set)')
print('-'*60)
print(f'{"Model":<22} {"MAE":>10} {"RMSE":>10} {"R²":>10}')
print('-'*60)
for name, v in table3.items():
    print(f'{name:<22} {v["MAE"]:>10.4f} {v["RMSE"]:>10.4f} {v["R2"]:>10.4f}')
print('='*60)

print('\nFailure Classification (FD001 Test Set):')
print(f'{"Model":<28} {"Precision":>10} {"Recall":>10} {"F1":>8} {"AUC":>8}')
print('-'*70)
for name, v in [('Random Forest', failrf_r), ('Gradient Boosting', failgb_r)]:
    print(f'{name:<28} {v["precision"]:>10.4f} {v["recall"]:>10.4f} {v["f1"]:>8.4f} {v["roc_auc"]:>8.4f}')

# Save
(OUT / 'table3_metrics.json').write_text(json.dumps(table3, indent=2))

all_metrics = {
    'dataset': 'FD001', 'train_samples': int(len(X_train)),
    'test_samples': 100, 'feature_count': len(FEAT_COLS),
    'rul_rf':   {k: v for k,v in rf_r.items()   if k != 'preds'},
    'rul_gb':   {k: v for k,v in gb_r.items()   if k != 'preds'},
    'rul_xgb':  {k: v for k,v in xgb_r.items()  if k != 'preds'},
    'lstm':     {k: v for k,v in lstm_r.items()  if k != 'preds'},
    'gru':      {k: v for k,v in gru_r.items()   if k != 'preds'},
    'fail_rf':  {k: v for k,v in failrf_r.items() if k not in ('preds','proba')},
    'fail_gb':  {k: v for k,v in failgb_r.items() if k not in ('preds','proba')},
    'production_monitoring': {
        'rul_below_30':       int(np.sum(rf_r['preds'] < 30)),
        'rul_below_15':       int(np.sum(rf_r['preds'] < 15)),
        'fail_prob_above_07': int(np.sum(failgb_r['proba'] > 0.7)),
    }
}
(MODELS / 'model_metrics_fd001.json').write_text(json.dumps(all_metrics, indent=2))
print('\n✓ Metrics saved')

## Cell 9 — Anomaly Detection (Isolation Forest)

In [ ]:
print('Training Isolation Forest ...')
X_anom    = train_raw[SENSORS].values
sc_anom   = StandardScaler().fit(X_anom)
X_anom_sc = sc_anom.transform(X_anom)

iso = IsolationForest(n_estimators=100, contamination=0.05,
                      random_state=42, n_jobs=-1)
iso.fit(X_anom_sc)

# Score test-set last cycles
X_te_sens  = test_last[SENSORS].values
X_te_asc   = sc_anom.transform(X_te_sens)
iso_scores = iso.decision_function(X_te_asc)
iso_labels = iso.predict(X_te_asc)

n_anom = int(np.sum(iso_labels == -1))
print(f'  Flagged anomalies (last cycle): {n_anom}/100')
print(f'  Normal mean score:  {iso_scores[iso_labels==1].mean():.4f}')
print(f'  Anomaly mean score: {iso_scores[iso_labels==-1].mean():.4f}  (lower = more anomalous)')

joblib.dump(iso,     MODELS / 'isolation_forest_fd001.joblib')
joblib.dump(sc_anom, MODELS / 'anomaly_scaler_fd001.joblib')
print('✓ Anomaly model saved')

## Cell 10 — Fig 2: Dataset Exploration

In [ ]:
DIAG_SENSORS = ['sensor2','sensor3','sensor4','sensor7','sensor11',
                 'sensor12','sensor15','sensor17']

fig = plt.figure(figsize=(18, 14), facecolor=BG)
fig.suptitle(
    'Fig 2 — Dataset Exploration & Sensor Measurements\n'
    'FD001 Training Set  |  100 Engines  |  20,631 Observations',
    fontsize=13, color=TXT, y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

lc = train_raw.groupby('unit')['cycle'].max()
ax0 = fig.add_subplot(gs[0, :2])
ax0.hist(lc, bins=20, color=PRI, edgecolor=BG, alpha=0.85)
ax0.axvline(lc.mean(), color=WRN, lw=1.5, ls='--', label=f'Mean {lc.mean():.0f}')
ax0.axvline(lc.median(), color=ACC, lw=1.5, ls=':', label=f'Median {lc.median():.0f}')
ax0.set_xlabel('Engine Lifecycle (cycles)'); ax0.set_ylabel('Count')
ax0.set_title('(a) Engine Lifecycle Distribution', color=TXT)
ax0.legend(fontsize=8); ax0.grid(True)

ax1 = fig.add_subplot(gs[0, 2:])
ax1.hist(rul_true, bins=20, color=ACC, edgecolor=BG, alpha=0.85)
ax1.axvline(rul_true.mean(), color=WRN, lw=1.5, ls='--', label=f'Mean {rul_true.mean():.0f}')
ax1.set_xlabel('True RUL at Test Cutoff (cycles)'); ax1.set_ylabel('Count')
ax1.set_title('(b) Test-Set True RUL Distribution', color=TXT)
ax1.legend(fontsize=8); ax1.grid(True)

positions = [(1,0),(1,1),(1,2),(1,3),(2,0),(2,1),(2,2),(2,3)]
colors    = [PRI,SEC,ACC,WRN,SUC,DNG,SEC,PRI]
for i, (s, pos) in enumerate(zip(DIAG_SENSORS, positions)):
    ax = fig.add_subplot(gs[pos[0], pos[1]])
    ax.hist(train_raw[s].dropna(), bins=25, color=colors[i], edgecolor=BG, alpha=0.85)
    ax.set_title(f'({chr(99+i)}) {s}', color=TXT, fontsize=9)
    ax.set_xlabel('Value', fontsize=7); ax.set_ylabel('Count', fontsize=7)
    ax.tick_params(labelsize=7); ax.grid(True)

plt.savefig(OUT/'fig2_dataset_exploration.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 2 saved')

## Cell 11 — Fig 3: Sensor Degradation Trends

In [ ]:
TREND_SENSORS  = ['sensor2','sensor3','sensor4','sensor11','sensor12','sensor15']
SAMPLE_ENGINES = [1, 5, 10, 20, 50]
COLORS_E       = [PRI, ACC, WRN, SEC, DNG]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)
fig.suptitle('Fig 3 — Sensor Degradation Trends Over Operating Cycles\n'
             'FD001 Training Set — 5 Sample Engines',
             fontsize=13, color=TXT)
for ax, sensor in zip(axes.flatten(), TREND_SENSORS):
    for eng, col in zip(SAMPLE_ENGINES, COLORS_E):
        edf = train_raw[train_raw['unit'] == eng]
        ax.plot(edf['cycle'], edf[sensor], color=col, lw=1.3, alpha=0.85, label=f'Eng {eng}')
    ax.set_title(sensor, color=TXT)
    ax.set_xlabel('Cycle'); ax.set_ylabel('Value')
    ax.legend(fontsize=7, ncol=3); ax.grid(True); ax.set_facecolor(BG2)
plt.tight_layout()
plt.savefig(OUT/'fig3_sensor_degradation.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 3 saved')

## Cell 12 — Fig 4: Actual vs Predicted RUL (all models)

In [ ]:
all_reg_results = [
    ('Random Forest',     rf_r,   PRI,  y_test),
    ('Gradient Boosting', gb_r,   ACC,  y_test),
    ('XGBoost',           xgb_r,  SEC,  y_test),
    ('LSTM',              lstm_r, WRN,  y_seq_te),
    ('GRU',               gru_r,  DNG,  y_seq_te),
]

fig, axes = plt.subplots(1, 5, figsize=(24, 5), facecolor=BG)
fig.suptitle('Fig 4 — Actual versus Predicted RUL  |  FD001 Test Set  |  100 Engines',
             fontsize=13, color=TXT)
for ax, (name, m, col, y_true) in zip(axes, all_reg_results):
    ax.set_facecolor(BG2)
    ax.scatter(y_true, m['preds'], color=col, alpha=0.65, s=30, edgecolors='none')
    lim = max(y_true.max(), m['preds'].max()) + 10
    ax.plot([0,lim],[0,lim], color=WRN, lw=1.5, ls='--', label='Perfect')
    ax.set_xlabel('True RUL'); ax.set_ylabel('Predicted RUL')
    ax.set_title(f'{name}\nMAE={m["mae"]:.2f}  R²={m["r2"]:.3f}', color=TXT)
    ax.legend(fontsize=7); ax.grid(True)
plt.tight_layout()
plt.savefig(OUT/'fig4_actual_vs_predicted.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 4 saved')

## Cell 13 — Fig 5: Model Performance Comparison

In [ ]:
model_names = list(table3.keys())
mae_v  = [table3[k]['MAE']  for k in model_names]
rmse_v = [table3[k]['RMSE'] for k in model_names]
r2_v   = [table3[k]['R2']   for k in model_names]
MCOLORS = [PRI, ACC, SEC, WRN, DNG]
short   = ['RF', 'GB', 'XGB', 'LSTM', 'GRU']

fig, axes = plt.subplots(1, 3, figsize=(16, 6), facecolor=BG)
fig.suptitle('Fig 5 — Model Performance Comparison\nFD001 Test Set — All 5 Models',
             fontsize=13, color=TXT)
for ax, (title, vals, best_fn) in zip(axes, [
    ('MAE  ↓ lower is better',  mae_v,  min),
    ('RMSE ↓ lower is better',  rmse_v, min),
    ('R²   ↑ higher is better', r2_v,   max),
]):
    ax.set_facecolor(BG2)
    best = best_fn(vals)
    cols = [WRN if v==best else MCOLORS[i] for i,v in enumerate(vals)]
    bars = ax.bar(short, vals, color=cols, edgecolor=BG, width=0.55)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8, color=TXT)
    ax.set_title(title, color=TXT); ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig(OUT/'fig5_model_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 5 saved')

## Cell 14 — Fig 6: Anomaly Detection Results

In [ ]:
ENGINE_DEMO = 42
edf_demo  = test_raw[test_raw['unit'] == ENGINE_DEMO].copy()
X_demo_sc = sc_anom.transform(edf_demo[SENSORS].values)
s_demo    = iso.decision_function(X_demo_sc)
l_demo    = iso.predict(X_demo_sc)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle(f'Fig 6 — Anomaly Detection Results  |  Isolation Forest  |  FD001\n'
             f'Fleet: {n_anom}/100 flagged  |  Contamination=5%  |  Thresholds: −0.10, −0.20',
             fontsize=12, color=TXT)

ax = axes[0]; ax.set_facecolor(BG2)
ax.hist(iso_scores[iso_labels==1],  bins=20, color=SUC, alpha=0.7, label='Normal')
ax.hist(iso_scores[iso_labels==-1], bins=10, color=DNG, alpha=0.85, label='Anomaly')
ax.axvline(-0.10, color=WRN, lw=1.5, ls='--', label='Unusual')
ax.axvline(-0.20, color=DNG, lw=1.5, ls=':',  label='High Anomaly')
ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Count')
ax.set_title('(a) Fleet Score Distribution', color=TXT)
ax.legend(fontsize=8); ax.grid(True)

ax = axes[1]; ax.set_facecolor(BG2)
nm = l_demo == 1
ax.plot(edf_demo['cycle'][nm],  s_demo[nm],  color=SUC, lw=1.2, label='Normal')
ax.scatter(edf_demo['cycle'][~nm], s_demo[~nm], color=DNG, s=35, zorder=5, label='Anomaly')
ax.axhline(-0.10, color=WRN, lw=1.2, ls='--', label='Unusual')
ax.axhline(-0.20, color=DNG, lw=1.2, ls=':',  label='High Anomaly')
ax.set_xlabel('Cycle'); ax.set_ylabel('Score')
ax.set_title(f'(b) Engine {ENGINE_DEMO} — Score Timeline', color=TXT)
ax.legend(fontsize=8); ax.grid(True)

ax = axes[2]; ax.set_facecolor(BG2)
sc = ax.scatter(y_test, iso_scores, c=iso_scores, cmap='RdYlGn',
                s=40, alpha=0.8, vmin=-0.3, vmax=0.2)
ax.axhline(-0.10, color=WRN, lw=1.2, ls='--', label='Unusual')
ax.axhline(-0.20, color=DNG, lw=1.2, ls=':',  label='High Anomaly')
ax.set_xlabel('True RUL (cycles)'); ax.set_ylabel('Anomaly Score')
ax.set_title('(c) Score vs True RUL', color=TXT)
ax.legend(fontsize=8); ax.grid(True)
plt.colorbar(sc, ax=ax, label='Score')

plt.tight_layout()
plt.savefig(OUT/'fig6_anomaly_detection.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 6 saved')

## Cell 15 — Fig 7: SHAP Analysis

In [ ]:
print('Computing SHAP values (TreeExplainer on Random Forest) ...')
t0 = time.time()
explainer   = shap.TreeExplainer(rf)
shap_vals   = explainer.shap_values(X_te_sc)   # (100, 150)
print(f'  SHAP computed in {time.time()-t0:.1f}s')

# Aggregate by base sensor
sensor_shap = {}
for j, feat in enumerate(FEAT_COLS):
    base = next((s for s in SENSORS if feat.startswith(s)), None)
    if base:
        sensor_shap[base] = sensor_shap.get(base, 0) + np.abs(shap_vals[:, j]).mean()
sensor_shap_sorted = sorted(sensor_shap.items(), key=lambda x: x[1], reverse=True)

top20_idx   = np.argsort(np.abs(shap_vals).mean(0))[::-1][:20]
top20_names = [FEAT_COLS[i] for i in top20_idx]
top20_vals  = np.abs(shap_vals).mean(0)[top20_idx]

print('\nTop 5 sensors by SHAP:')
for s, v in sensor_shap_sorted[:5]:
    print(f'  {s}: {v:.4f}')

COLORS_SHAP = [PRI if i<5 else SEC if i<10 else TXT2 for i in range(20)]
snames = [x[0] for x in sensor_shap_sorted]
svals  = [x[1] for x in sensor_shap_sorted]

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
fig.suptitle('Fig 7 — SHAP Sensor Contribution Analysis\n'
             'Random Forest · FD001 Test Set (100 engines)  |  Mean |SHAP| value',
             fontsize=13, color=TXT)

ax = axes[0]; ax.set_facecolor(BG2)
ax.barh(snames[::-1], svals[::-1],
        color=[PRI if v > np.median(svals) else SEC for v in svals[::-1]],
        edgecolor=BG)
ax.set_xlabel('Mean |SHAP| (aggregated)'); ax.set_title('(a) Sensor-Level Contribution', color=TXT)
ax.grid(True, axis='x')

ax = axes[1]; ax.set_facecolor(BG2)
ax.barh(range(20), top20_vals[::-1], color=COLORS_SHAP[::-1], edgecolor=BG)
ax.set_yticks(range(20)); ax.set_yticklabels(top20_names[::-1], fontsize=8)
ax.set_xlabel('Mean |SHAP|'); ax.set_title('(b) Top 20 Feature-Level SHAP', color=TXT)
ax.grid(True, axis='x')

plt.tight_layout()
plt.savefig(OUT/'fig7_shap_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

# Save ranking
(OUT/'shap_sensor_ranking.json').write_text(
    json.dumps({s: round(v,6) for s,v in sensor_shap_sorted}, indent=2))
print('✓ Fig 7 saved')

## Cell 16 — Fig 8: Health Monitoring Dashboard

In [ ]:
DEMO_ENGINES = [1, 10, 23, 42, 67, 88]
rf_preds_all  = np.maximum(rf.predict(X_te_sc), 0)
unit_list     = test_last['unit'].tolist()
top_sens_list = []
for i in range(len(test_last)):
    idx = FEAT_COLS[int(np.argmax(np.abs(shap_vals[i])))]
    top_sens_list.append(next((s for s in SENSORS if idx.startswith(s)), idx))

def health_status(rul):
    if rul>100: return ('Healthy',   SUC)
    if rul>50:  return ('Warning',   WRN)
    if rul>20:  return ('High Risk', '#FB923C')
    return ('Critical', DNG)

def anom_status(score):
    if score<-0.20: return ('High Anomaly', DNG)
    if score<-0.10: return ('Unusual',      WRN)
    return ('Normal', SUC)

fig = plt.figure(figsize=(20, 14), facecolor=BG)
fig.suptitle('Fig 8 — Machine Health Monitoring Dashboard\n'
             'AeroSentinel  |  FD001 Test Set  |  RF + Isolation Forest',
             fontsize=14, color=TXT, y=0.99)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

import matplotlib.patches as mpatches
for idx, eng_unit in enumerate(DEMO_ENGINES):
    row, col = divmod(idx, 3)
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor(SURF); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')

    ui = unit_list.index(eng_unit)
    pred_rul   = float(rf_preds_all[ui])
    true_rul   = float(rul_true.iloc[ui])
    a_score    = float(iso_scores[ui])
    h_lbl, h_c = health_status(pred_rul)
    a_lbl, a_c = anom_status(a_score)
    top_s       = top_sens_list[ui]
    last_cyc    = int(test_last['cycle'].iloc[ui])

    ax.text(0.5,0.94,f'ENGINE {eng_unit:03d}',ha='center',va='top',
            fontsize=11,fontweight='bold',color=TXT,transform=ax.transAxes)

    max_rul=200; frac=min(pred_rul/max_rul,1.0)
    theta=np.linspace(np.pi, np.pi-frac*np.pi, 100)
    ax.plot([0.28,0.72],[0.62,0.62],lw=6,color='#1E2D4A',
            solid_capstyle='round',transform=ax.transAxes)
    ax.plot(0.5+0.22*np.cos(theta), 0.62+0.20*np.sin(theta),
            lw=6, color=h_c, solid_capstyle='round', transform=ax.transAxes)
    ax.text(0.5,0.55,f'{pred_rul:.0f}',ha='center',va='center',
            fontsize=14,fontweight='bold',color=h_c,transform=ax.transAxes)
    ax.text(0.5,0.47,'cycles RUL',ha='center',va='center',
            fontsize=7,color=TXT2,transform=ax.transAxes)

    for bx,(lbl,col_b) in enumerate([(h_lbl,h_c),(a_lbl,a_c)]):
        r=mpatches.FancyBboxPatch((0.05+bx*0.5,0.33),0.42,0.1,
            boxstyle='round,pad=0.02',transform=ax.transAxes,
            facecolor=col_b+'25',edgecolor=col_b,lw=0.8)
        ax.add_patch(r)
        ax.text(0.26+bx*0.5,0.385,lbl,ha='center',va='center',
                fontsize=7,color=col_b,transform=ax.transAxes)

    for ii,(lbl,val) in enumerate([('Last Cycle',str(last_cyc)),
                                    ('Anom Score',f'{a_score:.3f}'),
                                    ('Top Sensor',top_s),
                                    ('True RUL',str(int(true_rul)))]):
        y_pos = 0.26-ii*0.07
        ax.text(0.05,y_pos,lbl+':',fontsize=7,color=TXT2,transform=ax.transAxes)
        ax.text(0.95,y_pos,val,fontsize=7,color=PRI if ii<2 else ACC,
                ha='right',transform=ax.transAxes,fontweight='bold')

plt.savefig(OUT/'fig8_health_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Fig 8 saved')

## Cell 17 — Training curves (LSTM + GRU)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=BG)
fig.suptitle('LSTM & GRU Training History — FD001\n'
             f'Window={WINDOW} · {N_FEAT} sensors · RUL cap={RUL_CAP}',
             fontsize=12, color=TXT)

for i, (hist, name, col) in enumerate([
    (lstm_hist, 'LSTM', SEC),
    (gru_hist,  'GRU',  WRN),
]):
    ax_loss = axes[i][0]; ax_loss.set_facecolor(BG2)
    ax_loss.plot(hist.history['loss'],     color=col, lw=1.8, label='Train MSE')
    ax_loss.plot(hist.history['val_loss'], color=DNG, lw=1.8, ls='--', label='Val MSE')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('MSE')
    ax_loss.set_title(f'{name} — Loss', color=TXT)
    ax_loss.legend(fontsize=9); ax_loss.grid(True)

    ax_mae = axes[i][1]; ax_mae.set_facecolor(BG2)
    ax_mae.plot(hist.history['mae'],     color=col, lw=1.8, label='Train MAE')
    ax_mae.plot(hist.history['val_mae'], color=DNG, lw=1.8, ls='--', label='Val MAE')
    ax_mae.set_xlabel('Epoch'); ax_mae.set_ylabel('MAE (cycles)')
    ax_mae.set_title(f'{name} — MAE', color=TXT)
    ax_mae.legend(fontsize=9); ax_mae.grid(True)

plt.tight_layout()
plt.savefig(OUT/'fig_training_curves.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Training curves saved')

## Cell 18 — Failure classifier visualisation (ROC + Confusion Matrix)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Failure Classification Results — FD001  |  Gradient Boosting (Preferred)',
             fontsize=13, color=TXT)

# Confusion matrix
cm = confusion_matrix(y_test_fail, failgb_r['preds'])
ax = axes[0]; ax.set_facecolor(BG2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Fail','Fail'], yticklabels=['No Fail','Fail'],
            cbar=False)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('(a) Confusion Matrix — GB Classifier', color=TXT)

# ROC curve
ax = axes[1]; ax.set_facecolor(BG2)
fpr, tpr, _ = roc_curve(y_test_fail, failgb_r['proba'])
ax.plot(fpr, tpr, color=ACC, lw=2,
        label=f'GB AUC={failgb_r["roc_auc"]:.3f}')
fpr2, tpr2, _ = roc_curve(y_test_fail, failrf_r['proba'])
ax.plot(fpr2, tpr2, color=PRI, lw=2, ls='--',
        label=f'RF AUC={failrf_r["roc_auc"]:.3f}')
ax.plot([0,1],[0,1], color=TXT2, lw=1, ls=':')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('(b) ROC Curve', color=TXT)
ax.legend(fontsize=9); ax.grid(True)

# Probability distribution
ax = axes[2]; ax.set_facecolor(BG2)
ax.hist(failgb_r['proba'][y_test_fail==0], bins=20, alpha=0.6, color=SUC, label='No Failure')
ax.hist(failgb_r['proba'][y_test_fail==1], bins=20, alpha=0.7, color=DNG, label='Failure')
ax.axvline(0.5, color=WRN, lw=1.5, ls='--', label='Threshold 0.5')
ax.axvline(0.7, color=DNG, lw=1.5, ls=':',  label='High Risk 0.7')
ax.set_xlabel('Predicted Probability'); ax.set_ylabel('Count')
ax.set_title('(c) Probability Distribution', color=TXT)
ax.legend(fontsize=8); ax.grid(True)

plt.tight_layout()
plt.savefig(OUT/'fig_failure_classification.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✓ Failure classification figure saved')

## Cell 19 — Print final summary

In [ ]:
print('='*65)
print('AEROSENTINEL — COMPLETE TRAINING RESULTS')
print('Dataset: FD001 | Train: 20,631 rows | Test: 100 engines')
print('='*65)

print(f'\n{'TABLE 3 — RUL REGRESSION':}')
print(f'{"Model":<22} {"MAE":>10} {"RMSE":>10} {"R²":>10}')
print('-'*55)
for k,v in table3.items():
    print(f'{k:<22} {v["MAE"]:>10.4f} {v["RMSE"]:>10.4f} {v["R2"]:>10.4f}')

print(f'\nFAILURE CLASSIFICATION (30-cycle window)')
print(f'{"Model":<22} {"Precision":>10} {"Recall":>10} {"F1":>8} {"AUC":>8}')
print('-'*62)
for name, v in [("Random Forest",failrf_r),("Gradient Boosting",failgb_r)]:
    print(f'{name:<22} {v["precision"]:>10.4f} {v["recall"]:>10.4f} {v["f1"]:>8.4f} {v["roc_auc"]:>8.4f}')

pm = all_metrics['production_monitoring']
print(f'\nPRODUCTION MONITORING (100 test engines)')
print(f'  RUL < 30 cycles     : {pm["rul_below_30"]}')
print(f'  RUL < 15 cycles     : {pm["rul_below_15"]}')
print(f'  Fail prob > 0.7     : {pm["fail_prob_above_07"]}')

print(f'\nSHAP TOP 5 SENSORS')
for s,v in sensor_shap_sorted[:5]:
    print(f'  {s}: {v:.4f}')

print(f'\nANOMALY DETECTION')
print(f'  Flagged at test cutoff: {n_anom}/100')
print(f'  Contamination param   : 5%')

print(f'\nFIGURES SAVED:')
for f in sorted(OUT.glob('*.png')):
    print(f'  {f.name}')

print(f'\nMODELS SAVED:')
for f in sorted(MODELS.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')
print('='*65)

## Cell 20 — Download everything as ZIP

In [ ]:
import shutil, zipfile

zip_path = 'AeroSentinel_Results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Figures
    for f in OUT.glob('*.png'):
        zf.write(f, f'figures/{f.name}')
    for f in OUT.glob('*.json'):
        zf.write(f, f'figures/{f.name}')
    # Metrics JSON
    zf.write(MODELS/'model_metrics_fd001.json', 'models/model_metrics_fd001.json')
    zf.write(MODELS/'feature_cols_fd001.json',  'models/feature_cols_fd001.json')
    # Note: joblib/keras files are large — include if needed
    # for f in MODELS.glob('*.joblib'): zf.write(f, f'models/{f.name}')
    # for f in MODELS.glob('*.keras'):  zf.write(f, f'models/{f.name}')

print(f'ZIP created: {zip_path}  ({Path(zip_path).stat().st_size/1024:.0f} KB)')

from google.colab import files
files.download(zip_path)
print('✓ Download started')